# OLMo-2-1124-7B-SFT + GRPO LoRA Reasoning RPT Evaluation

This notebook evaluates the **Reasoning RPT** model:

```text
base model: allenai/OLMo-2-1124-7B-SFT
LoRA adapter: checkpoint-450
```

on:

1. CommonsenseQA
2. IFEval generation
3. Official IFEval evaluator

For your project, this is the **Reasoning RPT** model if the LoRA adapter was trained on GSM8K with reasoning traces / step-by-step solutions.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 0: Install dependencies

Run this cell first in Colab or your GPU environment.

In [3]:
!pip install -U transformers accelerate datasets evaluate peft safetensors sentencepiece torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:


In [4]:
!pip install -q --upgrade transformers accelerate datasets evaluate sentencepiece tqdm peft safetensors bitsandbytes
!pip install -q absl-py immutabledict nltk langdetect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 67.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Step 1: Configuration

Set `ADAPTER_PATH` to the folder containing `adapter_config.json` and `adapter_model.safetensors`.

In [5]:
from pathlib import Path

# Dataset limits
CSQA_LIMIT = None          # None = full validation set
IFEVAL_FIRST_N = 100       # first batch for quick official evaluation
GENERATE_FULL_IFEVAL = True

# Model paths
BASE_MODEL_NAME = "allenai/OLMo-2-1124-7B-SFT"

# After uploading/unzipping checkpoint-450, this folder should contain:
# adapter_config.json and adapter_model.safetensors
ADAPTER_PATH = Path("/content/drive/MyDrive/olmo_grpo/checkpoint-450")

# Optional: if you put checkpoint-450.zip in Google Drive, set this path.
# Example: /content/drive/MyDrive/olmo_grpo/checkpoint-450.zip
ADAPTER_ZIP_PATH = Path("/content/drive/MyDrive/olmo_grpo/checkpoint-450.zip")

# Output directory for this Reasoning RPT run
OUTPUT_DIR = Path("reasoning_rpt_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_MODEL_NAME:", BASE_MODEL_NAME)
print("ADAPTER_PATH:", ADAPTER_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


BASE_MODEL_NAME: allenai/OLMo-2-1124-7B-SFT
ADAPTER_PATH: /content/drive/MyDrive/olmo_grpo/checkpoint-450
OUTPUT_DIR: reasoning_rpt_results


## Step 2: Upload / unzip LoRA adapter

Recommended Colab workflow:

1. Zip your local `checkpoint-450/` folder into `checkpoint-450.zip`.
2. Upload it to Google Drive, for example: `MyDrive/olmo_grpo/checkpoint-450.zip`.
3. Run this cell to mount Drive and unzip it.

If you manually uploaded the folder to `/content/olmo_grpo/checkpoint-450`, this cell will simply check the files.

In [6]:
import os
from pathlib import Path

# Mount Google Drive in Colab
from google.colab import drive
drive.mount("/content/drive")

# Direct folder path in Google Drive
ADAPTER_PATH = Path("/content/drive/MyDrive/olmo_grpo/checkpoint-450")

print("Checking adapter folder:", ADAPTER_PATH)
print("ADAPTER_PATH exists:", ADAPTER_PATH.exists())
print("adapter_config.json exists:", (ADAPTER_PATH / "adapter_config.json").exists())
print("adapter_model.safetensors exists:", (ADAPTER_PATH / "adapter_model.safetensors").exists())

if ADAPTER_PATH.exists():
    print("Files:", os.listdir(ADAPTER_PATH))
else:
    raise FileNotFoundError(f"ADAPTER_PATH does not exist: {ADAPTER_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking adapter folder: /content/drive/MyDrive/olmo_grpo/checkpoint-450
ADAPTER_PATH exists: True
adapter_config.json exists: True
adapter_model.safetensors exists: True
Files: ['adapter_config.json', 'adapter_model.safetensors', 'training_args.bin', 'tokenizer.json', 'chat_template.jinja', 'tokenizer_config.json', 'README.md', 'optimizer.pt', 'rng_state.pth', 'scheduler.pt', 'trainer_state.json']


## Step 3: Load Reasoning RPT model

This loads:

```text
allenai/OLMo-2-1124-7B-SFT + checkpoint-450 LoRA adapter
```

Do **not** use `AutoModelForCausalLM.from_pretrained(checkpoint-450)` directly, because `checkpoint-450` is only a LoRA adapter, not a full 7B model.


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

LOAD_IN_8BIT = False   # Set True if Colab GPU memory is not enough.

offload_dir = Path("offload")
offload_dir.mkdir(exist_ok=True)

# Tokenizer: prefer adapter tokenizer if present; otherwise use base tokenizer.
try:
    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_PATH))
    print("Loaded tokenizer from adapter path.")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    print("Loaded tokenizer from base model.")

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

if LOAD_IN_8BIT:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
        offload_folder=str(offload_dir),
        low_cpu_mem_usage=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        dtype=torch.float16,
        device_map="auto",
        offload_folder=str(offload_dir),
        low_cpu_mem_usage=True,
    )

model = PeftModel.from_pretrained(
    base_model,
    str(ADAPTER_PATH),
)

model.eval()

print("Loaded Reasoning RPT model:")
print("Base:", BASE_MODEL_NAME)
print("Adapter:", ADAPTER_PATH)


Loaded tokenizer from adapter path.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/675 [00:00<?, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

Loaded Reasoning RPT model:
Base: allenai/OLMo-2-1124-7B-SFT
Adapter: /content/drive/MyDrive/olmo_grpo/checkpoint-450


## Step 4: Define generation helper

We use one shared `generate_text()` function for CommonsenseQA and IFEval.

In [8]:
import re
import json
from typing import Dict, Any, Optional
from datasets import load_dataset
from tqdm.auto import tqdm

def get_first_model_device(model):
    for p in model.parameters():
        if p.device.type != "meta":
            return p.device
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def generate_text(
    prompt: str,
    max_new_tokens: int = 128,
    temperature: float = 0.0,
) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_token_type_ids=False,
    )

    # Do not use model.device directly because device_map="auto" may shard/offload the model.
    first_device = get_first_model_device(model)
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    do_sample = temperature > 0

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if decoded.startswith(prompt):
        return decoded[len(prompt):].strip()
    return decoded.strip()


# Part A: CommonsenseQA Reasoning RPT Evaluation

We evaluate on the CommonsenseQA validation split because it has gold answers.

In [9]:
csqa = load_dataset("tau/commonsense_qa", split="validation")

if CSQA_LIMIT is not None:
    csqa = csqa.select(range(min(CSQA_LIMIT, len(csqa))))

print(csqa)
print(csqa[0])


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
    num_rows: 1221
})
{'id': '1afa02df02c908a558b4036e80242fac', 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'question_concept': 'revolving door', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}, 'answerKey': 'A'}


In [10]:
def build_csqa_prompt(ex: Dict[str, Any]) -> str:
    labels = ex["choices"]["label"]
    texts = ex["choices"]["text"]
    options = "\n".join([f"{label}. {text}" for label, text in zip(labels, texts)])

    return (
        "Answer the following multiple-choice question.\n"
        "Return only one letter: A, B, C, D, or E.\n\n"
        f"Question: {ex['question']}\n"
        f"{options}\n\n"
        "Answer:"
    )


def extract_csqa_answer(text: str) -> Optional[str]:
    text = text.strip().upper()

    # First, parse outputs that start with A-E.
    match = re.match(r"^\s*([ABCDE])\b", text)
    if match:
        return match.group(1)

    # Parse "Answer: A" format.
    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCDE])", text)
    if match:
        return match.group(1)

    # Fallback: any standalone A-E.
    match = re.search(r"\b([ABCDE])\b", text)
    if match:
        return match.group(1)

    return None


In [11]:
csqa_predictions = []
correct = 0
parsed = 0

for ex in tqdm(csqa, desc="Evaluating CommonsenseQA"):
    prompt = build_csqa_prompt(ex)

    raw_output = generate_text(
        prompt,
        max_new_tokens=8,
        temperature=0.0,
    )

    pred = extract_csqa_answer(raw_output)
    gold = ex["answerKey"]

    if pred is not None:
        parsed += 1

    if pred == gold:
        correct += 1

    csqa_predictions.append({
        "id": ex["id"],
        "question": ex["question"],
        "gold": gold,
        "prediction": pred,
        "raw_output": raw_output,
    })

csqa_accuracy = correct / len(csqa)
csqa_parse_rate = parsed / len(csqa)

csqa_metrics = {
    "model_type": "Reasoning RPT",
    "base_model": BASE_MODEL_NAME,
    "adapter_path": str(ADAPTER_PATH),
    "benchmark": "CommonsenseQA",
    "split": "validation",
    "num_examples": len(csqa),
    "accuracy": csqa_accuracy,
    "parse_rate": csqa_parse_rate,
}

print(json.dumps(csqa_metrics, indent=2))

with open(OUTPUT_DIR / "commonsenseqa_predictions.json", "w", encoding="utf-8") as f:
    json.dump(csqa_predictions, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "commonsenseqa_metrics.json", "w", encoding="utf-8") as f:
    json.dump(csqa_metrics, f, ensure_ascii=False, indent=2)

print("Saved:", OUTPUT_DIR / "commonsenseqa_predictions.json")
print("Saved:", OUTPUT_DIR / "commonsenseqa_metrics.json")

# Inspect first 5 outputs to check parsing.
for item in csqa_predictions[:5]:
    print("=" * 80)
    print("Gold:", item["gold"])
    print("Pred:", item["prediction"])
    print("Raw:", item["raw_output"])


Evaluating CommonsenseQA:   0%|          | 0/1221 [00:00<?, ?it/s]

{
  "model_type": "Reasoning RPT",
  "base_model": "allenai/OLMo-2-1124-7B-SFT",
  "adapter_path": "/content/drive/MyDrive/olmo_grpo/checkpoint-450",
  "benchmark": "CommonsenseQA",
  "split": "validation",
  "num_examples": 1221,
  "accuracy": 0.6961506961506961,
  "parse_rate": 1.0
}
Saved: reasoning_rpt_results/commonsenseqa_predictions.json
Saved: reasoning_rpt_results/commonsenseqa_metrics.json
Gold: A
Pred: A
Raw: A. bank
Gold: A
Pred: A
Raw: A.
Gold: B
Pred: B
Raw: B.
Gold: A
Pred: A
Raw: A.
Gold: A
Pred: A
Raw: A.


# Part B: IFEval Reasoning RPT Generation

IFEval has two stages:

1. Generate model responses.
2. Use the official evaluator to check whether each response follows the instruction constraints.

This notebook saves model generations first, then converts them into the official evaluator input format.


## Generate first 100 IFEval responses

This is useful for quick testing and matches your baseline notebook's first-100 evaluation flow.

In [12]:
ifeval_full = load_dataset("google/IFEval", split="train")

ifeval_first = ifeval_full.select(range(min(IFEVAL_FIRST_N, len(ifeval_full))))

print(ifeval_first)
print(ifeval_first[0])

ifeval_generations = []

for idx, ex in enumerate(tqdm(ifeval_first, desc="Generating first IFEval responses")):
    prompt = ex["prompt"]

    response = generate_text(
        prompt,
        max_new_tokens=512,
        temperature=0.0,
    )

    ifeval_generations.append({
        "original_index": idx,
        "key": ex.get("key", None),
        "prompt": prompt,
        "instruction_id_list": ex["instruction_id_list"],
        "kwargs": ex["kwargs"],
        "response": response,
    })

output_file = OUTPUT_DIR / "ifeval_generations.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ifeval_generations, f, ensure_ascii=False, indent=2)

print("Saved:", output_file)
print("Number of examples:", len(ifeval_generations))
print("Example response:")
print(ifeval_generations[0]["response"][:1000])


README.md: 0.00B [00:00, ?B/s]

ifeval_input_data.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/541 [00:00<?, ? examples/s]

Dataset({
    features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
    num_rows: 100
})
{'key': 1000, 'prompt': 'Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*.', 'instruction_id_list': ['punctuation:no_comma', 'detectable_format:number_highlighted_sections', 'length_constraints:number_words'], 'kwargs': [{'num_highlights': None, 'relation': None, 'num_words': None, 'num_placeholders': None, 'prompt_to_repeat': None, 'num_bullets': None, 'section_spliter': None, 'num_sections': None, 'capital_relation': None, 'capital_frequency': None, 'keywords': None, 'num_paragraphs': None, 'language': None, 'let_relation': None, 'letter': None, 'let_frequency': None, 'end_phrase': None, 'forbidden_words': None, 'keyword': None, 'frequenc

Generating first IFEval responses:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Optional: Generate remaining IFEval responses and merge full 541

Run this if you want full official IFEval results instead of only the first 100.

In [13]:
if GENERATE_FULL_IFEVAL:
    IFEVAL_START = IFEVAL_FIRST_N
    IFEVAL_END = len(ifeval_full)

    ifeval_remaining = ifeval_full.select(range(IFEVAL_START, IFEVAL_END))

    ifeval_generations_remaining = []

    for idx, ex in enumerate(tqdm(ifeval_remaining, desc="Generating remaining IFEval responses")):
        prompt = ex["prompt"]

        response = generate_text(
            prompt,
            max_new_tokens=512,
            temperature=0.0,
        )

        ifeval_generations_remaining.append({
            "original_index": IFEVAL_START + idx,
            "key": ex.get("key", None),
            "prompt": prompt,
            "instruction_id_list": ex["instruction_id_list"],
            "kwargs": ex["kwargs"],
            "response": response,
        })

    remaining_file = OUTPUT_DIR / "ifeval_generations_remaining_100_540.json"
    with open(remaining_file, "w", encoding="utf-8") as f:
        json.dump(ifeval_generations_remaining, f, ensure_ascii=False, indent=2)

    full_541 = ifeval_generations + ifeval_generations_remaining
    full_file = OUTPUT_DIR / "ifeval_generations_full_541.json"
    with open(full_file, "w", encoding="utf-8") as f:
        json.dump(full_541, f, ensure_ascii=False, indent=2)

    print("Saved:", remaining_file)
    print("Saved:", full_file)
    print("Full examples:", len(full_541))
else:
    print("Skipping full IFEval generation.")


Generating remaining IFEval responses:   0%|          | 0/441 [00:00<?, ?it/s]

Saved: reasoning_rpt_results/ifeval_generations_remaining_100_540.json
Saved: reasoning_rpt_results/ifeval_generations_full_541.json
Full examples: 451


In [14]:
import json
from pathlib import Path

first_path = Path("reasoning_rpt_results/ifeval_generations.json")
remaining_path = Path("reasoning_rpt_results/ifeval_generations_remaining_100_540.json")

output_path = Path("reasoning_rpt_results/ifeval_generations_full_541.json")

with open(first_path, "r", encoding="utf-8") as f:
    first_100 = json.load(f)

with open(remaining_path, "r", encoding="utf-8") as f:
    remaining_441 = json.load(f)

print("First file examples:", len(first_100))
print("Remaining file examples:", len(remaining_441))

merged = first_100 + remaining_441

# 去重：如果某些 original_index 重复，只保留最后一次出现的
by_index = {}
for row in merged:
    by_index[row["original_index"]] = row

merged = list(by_index.values())

# 按 original_index 排序
merged = sorted(merged, key=lambda x: x["original_index"])

print("Merged examples:", len(merged))
print("First index:", merged[0]["original_index"])
print("Last index:", merged[-1]["original_index"])

# 检查是否刚好是 0 到 540
indices = [row["original_index"] for row in merged]
missing = sorted(set(range(541)) - set(indices))
duplicates = len(indices) - len(set(indices))

print("Missing indices:", missing[:20])
print("Number of missing:", len(missing))
print("Number of duplicate indices:", duplicates)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print("Saved full merged IFEval generations to:", output_path)

First file examples: 100
Remaining file examples: 441
Merged examples: 541
First index: 0
Last index: 540
Missing indices: []
Number of missing: 0
Number of duplicate indices: 0
Saved full merged IFEval generations to: reasoning_rpt_results/ifeval_generations_full_541.json


## Step B2: Clone official IFEval evaluator

In [15]:
import os
from pathlib import Path

if not Path("google-research").exists():
    !git clone --depth 1 https://github.com/google-research/google-research.git
else:
    print("google-research already exists.")


Cloning into 'google-research'...
remote: Enumerating objects: 23255, done.
remote: Counting objects: 100% (23255/23255), done.
remote: Compressing objects: 100% (18114/18114), done.
remote: Total 23255 (delta 4638), reused 14515 (delta 3954), pack-reused 0 (from 0)
Receiving objects: 100% (23255/23255), 481.64 MiB | 18.50 MiB/s, done.
Resolving deltas: 100% (4638/4638), done.
Updating files: 100% (22118/22118), done.


## Step B3: Convert generated responses into official IFEval JSONL format

In [16]:
def clean_dict(d):
    """Remove keys whose values are None."""
    if d is None:
        return {}
    return {k: v for k, v in d.items() if v is not None}


def normalize_ifeval_kwargs(kwargs, instruction_id_list):
    """
    Convert HF google/IFEval kwargs to official evaluator format.

    Official format:
    kwargs = [
      {... kwargs for instruction 0 ...},
      {... kwargs for instruction 1 ...}
    ]

    Important:
    remove all None values.
    """
    n = len(instruction_id_list)

    if kwargs is None:
        return [{} for _ in range(n)]

    # Case 1: already list-of-dicts
    if isinstance(kwargs, list):
        fixed = []
        for item in kwargs:
            if isinstance(item, dict):
                fixed.append(clean_dict(item))
            else:
                fixed.append({})
        return fixed

    # Case 2: dict-of-lists
    if isinstance(kwargs, dict):
        fixed = [{} for _ in range(n)]

        for key, value in kwargs.items():
            if isinstance(value, list):
                for i in range(min(n, len(value))):
                    if value[i] is not None:
                        fixed[i][key] = value[i]
            else:
                if value is not None:
                    for i in range(n):
                        fixed[i][key] = value

        fixed = [clean_dict(x) for x in fixed]
        return fixed

    return [{} for _ in range(n)]


def write_ifeval_jsonl(generations, input_path, response_path):
    with open(input_path, "w", encoding="utf-8") as fin, \
         open(response_path, "w", encoding="utf-8") as fout:

        for i, ex in enumerate(generations):
            key = ex.get("key", None)
            if key is None:
                key = ex.get("original_index", i)

            instruction_id_list = ex["instruction_id_list"]
            fixed_kwargs = normalize_ifeval_kwargs(ex["kwargs"], instruction_id_list)

            input_obj = {
                "key": key,
                "instruction_id_list": instruction_id_list,
                "prompt": ex["prompt"],
                "kwargs": fixed_kwargs,
            }

            response_obj = {
                "key": key,
                "prompt": ex["prompt"],
                "response": ex["response"],
            }

            fin.write(json.dumps(input_obj, ensure_ascii=False) + "\n")
            fout.write(json.dumps(response_obj, ensure_ascii=False) + "\n")

    print("Wrote:", input_path)
    print("Wrote:", response_path)
    print("Num examples:", len(generations))


# First-100 files
with open(OUTPUT_DIR / "ifeval_generations.json", "r", encoding="utf-8") as f:
    generations_100 = json.load(f)

write_ifeval_jsonl(
    generations_100,
    OUTPUT_DIR / "ifeval_input_100.jsonl",
    OUTPUT_DIR / "ifeval_response_100.jsonl",
)


Wrote: reasoning_rpt_results/ifeval_input_100.jsonl
Wrote: reasoning_rpt_results/ifeval_response_100.jsonl
Num examples: 100


## Step B4: Run official IFEval evaluator for first 100

In [17]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [18]:
!rm -rf reasoning_rpt_results/ifeval_official_100
!mkdir -p reasoning_rpt_results/ifeval_official_100

!PYTHONPATH=google-research \
python google-research/instruction_following_eval/evaluation_main.py \
  --input_data=reasoning_rpt_results/ifeval_input_100.jsonl \
  --input_response_data=reasoning_rpt_results/ifeval_response_100.jsonl \
  --output_dir=reasoning_rpt_results/ifeval_official_100


I0428 19:08:34.689479 138926618034816 evaluation_main.py:57] Generating eval_results_strict...
I0428 19:08:35.171718 138926618034816 evaluation_main.py:63] Accuracy: 0.340000
I0428 19:08:35.174028 138926618034816 evaluation_main.py:69] Generated: reasoning_rpt_results/ifeval_official_100/eval_results_strict.jsonl
reasoning_rpt_results/ifeval_official_100/eval_results_strict.jsonl Accuracy Scores:
prompt-level: 0.34
instruction-level: 0.4723926380368098

change_case 0.47368421052631576
combination 0.09090909090909091
detectable_content 0.8888888888888888
detectable_format 0.4482758620689655
keywords 0.46153846153846156
language 0.75
length_constraints 0.2413793103448276
punctuation 0.8333333333333334
startend 0.7272727272727273

change_case:capital_word_frequency 0.5
change_case:english_capital 0.5
change_case:english_lowercase 0.45454545454545453
combination:repeat_prompt 0.0
combination:two_responses 0.25
detectable_content:number_placeholders 0.6666666666666666
detectable_content:pos

## Step B5: If full 541 generations exist, run official IFEval evaluator for full set

In [19]:
full_generation_path = OUTPUT_DIR / "ifeval_generations_full_541.json"

if full_generation_path.exists():
    with open(full_generation_path, "r", encoding="utf-8") as f:
        generations_full = json.load(f)

    write_ifeval_jsonl(
        generations_full,
        OUTPUT_DIR / "ifeval_input_full_541.jsonl",
        OUTPUT_DIR / "ifeval_response_full_541.jsonl",
    )
else:
    print("Full 541 generations not found. Skipping full JSONL conversion.")


Wrote: reasoning_rpt_results/ifeval_input_full_541.jsonl
Wrote: reasoning_rpt_results/ifeval_response_full_541.jsonl
Num examples: 541


In [20]:
if (OUTPUT_DIR / "ifeval_input_full_541.jsonl").exists():
    !rm -rf reasoning_rpt_results/ifeval_official_full_541
    !mkdir -p reasoning_rpt_results/ifeval_official_full_541

    !PYTHONPATH=google-research \
    python google-research/instruction_following_eval/evaluation_main.py \
      --input_data=reasoning_rpt_results/ifeval_input_full_541.jsonl \
      --input_response_data=reasoning_rpt_results/ifeval_response_full_541.jsonl \
      --output_dir=reasoning_rpt_results/ifeval_official_full_541
else:
    print("Full JSONL files not found. Skipping full official evaluation.")


I0428 19:09:30.428774 132996935672448 evaluation_main.py:57] Generating eval_results_strict...
E0428 19:09:30.977442 132996935672448 instructions.py:161] Unable to detect language for text " due to No features in text.
I0428 19:09:31.063449 132996935672448 evaluation_main.py:63] Accuracy: 0.395564
I0428 19:09:31.074001 132996935672448 evaluation_main.py:69] Generated: reasoning_rpt_results/ifeval_official_full_541/eval_results_strict.jsonl
reasoning_rpt_results/ifeval_official_full_541/eval_results_strict.jsonl Accuracy Scores:
prompt-level: 0.3955637707948244
instruction-level: 0.4880095923261391

change_case 0.449438202247191
combination 0.12307692307692308
detectable_content 0.8301886792452831
detectable_format 0.5222929936305732
keywords 0.48466257668711654
language 0.7419354838709677
length_constraints 0.36363636363636365
punctuation 0.5909090909090909
startend 0.5970149253731343

change_case:capital_word_frequency 0.56
change_case:english_capital 0.32
change_case:english_lowercas

## Step B6: Read official IFEval metrics

In [21]:
def read_ifeval_strict_metrics(official_dir: Path):
    strict_file = official_dir / "eval_results_strict.jsonl"

    records = []
    with open(strict_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    prompt_correct = 0
    instruction_correct = 0
    instruction_total = 0

    for r in records:
        follow_instruction_list = r["follow_instruction_list"]

        if all(follow_instruction_list):
            prompt_correct += 1

        instruction_correct += sum(follow_instruction_list)
        instruction_total += len(follow_instruction_list)

    return {
        "num_examples": len(records),
        "strict_prompt_accuracy": prompt_correct / len(records),
        "strict_instruction_accuracy": instruction_correct / instruction_total,
    }


ifeval_100_metrics = read_ifeval_strict_metrics(OUTPUT_DIR / "ifeval_official_100")
print("IFEval first 100 strict metrics:")
print(json.dumps(ifeval_100_metrics, indent=2))

ifeval_full_metrics = None
full_official_dir = OUTPUT_DIR / "ifeval_official_full_541"
if full_official_dir.exists():
    ifeval_full_metrics = read_ifeval_strict_metrics(full_official_dir)
    print("IFEval full 541 strict metrics:")
    print(json.dumps(ifeval_full_metrics, indent=2))


IFEval first 100 strict metrics:
{
  "num_examples": 100,
  "strict_prompt_accuracy": 0.34,
  "strict_instruction_accuracy": 0.4723926380368098
}
IFEval full 541 strict metrics:
{
  "num_examples": 541,
  "strict_prompt_accuracy": 0.3955637707948244,
  "strict_instruction_accuracy": 0.4880095923261391
}


## Step C: Save final summary JSON

In [22]:
summary = {
    "model_type": "Reasoning RPT",
    "base_model": BASE_MODEL_NAME,
    "adapter_path": str(ADAPTER_PATH),
    "commonsenseqa": {
        "split": "validation",
        "num_examples": len(csqa),
        "accuracy": csqa_accuracy,
        "parse_rate": csqa_parse_rate,
    },
    "ifeval_official_100": {
        "split": "train",
        **ifeval_100_metrics,
    },
}

if ifeval_full_metrics is not None:
    summary["ifeval_official_full_541"] = {
        "split": "train",
        **ifeval_full_metrics,
    }

summary_path = OUTPUT_DIR / "reasoning_rpt_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:", summary_path)
print(json.dumps(summary, indent=2))


Saved: reasoning_rpt_results/reasoning_rpt_summary.json
{
  "model_type": "Reasoning RPT",
  "base_model": "allenai/OLMo-2-1124-7B-SFT",
  "adapter_path": "/content/drive/MyDrive/olmo_grpo/checkpoint-450",
  "commonsenseqa": {
    "split": "validation",
    "num_examples": 1221,
    "accuracy": 0.6961506961506961,
    "parse_rate": 1.0
  },
  "ifeval_official_100": {
    "split": "train",
    "num_examples": 100,
    "strict_prompt_accuracy": 0.34,
    "strict_instruction_accuracy": 0.4723926380368098
  },
  "ifeval_official_full_541": {
    "split": "train",
    "num_examples": 541,
    "strict_prompt_accuracy": 0.3955637707948244,
    "strict_instruction_accuracy": 0.4880095923261391
  }
}
